# ElectInfo bootstraps its reporting environment

## What this shows
How to load a layered Hydra configuration, validate it with a Pydantic model, override one field via env var, and catch a bad override at load time. Uses ElectInfo — one of two partner firms that thread through these demos — onboarding a new TX analysis engagement.

## Why it matters
Every other capability in `siege_utilities` reads settings (cache dirs, API keys, output preferences) from the config layer. Getting this right once means the rest of the library uses sensible, client-scoped defaults automatically — no config re-plumbing inside each downstream notebook.

## Prereqs
- `pip install 'siege-utilities[config]'`
- No credentials.

## Next
- `02_profiles_branding.ipynb` — register ElectInfo and Masai Interactive as branded client profiles.


## 1. Load the shipped base configuration

`HydraConfigManager` resolves to `siege_utilities/configs/` by default. `load_config` composes YAML files under that root and returns a plain dict we can feed to a validator in the next cell.

In [1]:
from siege_utilities.config.hydra_manager import HydraConfigManager

mgr = HydraConfigManager()
print('config directory :', mgr.config_dir)

base = mgr.load_config('default/user_profile')
print('\ntop-level keys   :', list(base.keys()))

[siege_utilities] 2026-04-24 17:09:21,747 INFO: Initialized HydraConfigManager with config directory: /Users/dheerajchand/Documents/Professional/Siege_Analytics/Code/siege_utilities/siege_utilities/configs


config directory : /Users/dheerajchand/Documents/Professional/Siege_Analytics/Code/siege_utilities/siege_utilities/configs

top-level keys   : ['default']


## 2. Compose with an ElectInfo client overlay

Hydra overrides let ElectInfo ship one base profile and stamp client-specific deltas on top. Here we pull the shipped `client_a` overlay as a stand-in — when the ElectInfo client YAML lands it plugs in the same way.

In [2]:
from hydra.core.global_hydra import GlobalHydra

# Hydra allows only one init per process; reset before a second compose.
GlobalHydra.instance().clear()

mgr = HydraConfigManager()
overrides = [
    'default.organization=ElectInfo',
    'default.default_output_format=pdf',
    'default.default_dpi=200',
]
client_cfg = mgr.load_config('default/user_profile', overrides=overrides)

# Show only the keys that differ from the base so the overlay intent is clear.
diff = {
    k: v for k, v in client_cfg['default'].items()
    if base['default'].get(k) != v
}
print('overlaid fields :', diff)


[siege_utilities] 2026-04-24 17:09:21,775 INFO: Initialized HydraConfigManager with config directory: /Users/dheerajchand/Documents/Professional/Siege_Analytics/Code/siege_utilities/siege_utilities/configs


overlaid fields : {'organization': 'ElectInfo', 'default_output_format': 'pdf', 'default_dpi': 200}


## 3. Override a single field via env var

For one-off runs ("put today's downloads on the fast disk, just for this job") env vars are lighter than a YAML change. The library's on-disk cache helper reads `SIEGE_UTILITIES_CACHE_DIR` — flip it and the cache relocates without editing any files.

In [3]:
import os
from pathlib import Path
from siege_utilities.cache import _default_cache_dir

print('default cache dir :', _default_cache_dir())

os.environ['SIEGE_UTILITIES_CACHE_DIR'] = str(Path.cwd() / 'tmp_electinfo_cache')
print('overridden to     :', _default_cache_dir())

# Clean up so we don't leak the override into later notebooks in the same session.
del os.environ['SIEGE_UTILITIES_CACHE_DIR']

default cache dir : /Users/dheerajchand/.cache/siege_utilities/notebooks
overridden to     : /Users/dheerajchand/Documents/Professional/Siege_Analytics/Code/siege_utilities/notebooks/foundations/tmp_electinfo_cache


## 4. Validate a config dict with a Pydantic model

Pydantic models live under `siege_utilities.config.models`. `ReportPreferences` is a small, illustrative target — the same pattern applies to every other model the library ships (Person, Client, Credential, DatabaseConnection, …).

In [4]:
from siege_utilities.config.models.report_preferences import ReportPreferences

good = ReportPreferences(
    default_format='pdf',
    page_size='Letter',
    chart_dpi=300,
    chart_style='professional',
    include_executive_summary=True,
)
good.model_dump()


{'default_format': <ReportFormat.PDF: 'pdf'>,
 'include_executive_summary': True,
 'chart_style': 'professional',
 'page_size': 'Letter',
 'orientation': <PageOrientation.LANDSCAPE: 'landscape'>,
 'include_table_of_contents': True,
 'include_page_numbers': True,
 'include_watermark': False,
 'watermark_text': None,
 'chart_quality': 'high',
 'chart_dpi': 300,
 'include_chart_titles': True,
 'include_chart_legends': True,
 'include_data_labels': False,
 'sections': ['executive_summary',
  'methodology',
  'findings',
  'recommendations'],
 'export_metadata': True,
 'compress_output': False,
 'include_source_data': False}

## 5. Catching a bad override

If someone on the team mistypes an override value in YAML or a CI job, validation should fail loudly — not silently fall through to a broken report at 2am. Pydantic's `ValidationError` carries which field was wrong, so this is easy to surface in logs or error notifications.

In [5]:
from pydantic import ValidationError

try:
    # chart_dpi must be 72–600; 42 is rejected.
    ReportPreferences(chart_dpi=42)
except ValidationError as e:
    bad = [(err['loc'][0], err['msg']) for err in e.errors()]
    print(f'caught ValidationError on : {bad}')


caught ValidationError on : [('chart_dpi', 'Input should be greater than or equal to 72')]


## Related

- **Source**: `siege_utilities/config/` — `HydraConfigManager`, Pydantic models under `config/models/`
- **Tests**: `tests/test_config*.py`
- **Docs**: `docs/ARCHITECTURE.md` for the three-layer model; `docs/adr/0004-dataclass-vs-pydantic.md` for why we use Pydantic here
- **Next notebook**: `02_profiles_branding.ipynb` picks up from this config foundation to register ElectInfo + Masai Interactive as branded client profiles.
